In [108]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [109]:
data = pd.read_csv("TOTAL_KSI_3737821728629277523.csv")
print(data.shape)

(18957, 54)


In [ ]:
data_new = data.drop(['OBJECTID','INDEX', 'ACCNUM', 'STREET1', 'STREET2', 'OFFSET', 'LATITUDE', 
                  'LONGITUDE', 'INJURY', 'FATAL_NO', 'VEHTYPE', 'HOOD_158', 'HOOD_140', 
                  'NEIGHBOURHOOD_140', 'DIVISION', 'x', 'y', 'NEIGHBOURHOOD_158'], axis=1)
print(data_new.columns)

Index(['DATE', 'TIME', 'ROAD_CLASS', 'DISTRICT', 'ACCLOC', 'TRAFFCTL',
       'VISIBILITY', 'LIGHT', 'RDSFCOND', 'ACCLASS', 'IMPACTYPE', 'INVTYPE',
       'INVAGE', 'INITDIR', 'MANOEUVER', 'DRIVACT', 'DRIVCOND', 'PEDTYPE',
       'PEDACT', 'PEDCOND', 'CYCLISTYPE', 'CYCACT', 'CYCCOND', 'PEDESTRIAN',
       'CYCLIST', 'AUTOMOBILE', 'MOTORCYCLE', 'TRUCK', 'TRSN_CITY_VEH',
       'EMERG_VEH', 'PASSENGER', 'SPEEDING', 'AG_DRIV', 'REDLIGHT', 'ALCOHOL',
       'DISABILITY'],
      dtype='object')


In [111]:
X = data_new.drop("ACCLASS", axis=1)
y = data['ACCLASS']

In [112]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Train set size: (15165, 35)
Test set size: (3792, 35)


In [113]:
cols_with_nan = X_train.columns[X_train.isnull().any()].tolist()
nan_types = X_train[cols_with_nan].dtypes

In [114]:
fill_values = {
    col: 'No' if 'Yes' in X_train[col].unique() else 'Unknown' 
    for col in cols_with_nan
}

X_train = X_train.fillna(value=fill_values)
X_test = X_test.fillna(value=fill_values)

print(f"NaN in Train: {X_train.isnull().sum().sum()}")
print(f"NaN in Test: {X_test.isnull().sum().sum()}")

NaN in Train: 0
NaN in Test: 0


In [115]:
display(X_train.head())

,DATE,TIME,ROAD_CLASS,DISTRICT,ACCLOC,TRAFFCTL,VISIBILITY,LIGHT,RDSFCOND,IMPACTYPE,...,MOTORCYCLE,TRUCK,TRSN_CITY_VEH,EMERG_VEH,PASSENGER,SPEEDING,AG_DRIV,REDLIGHT,ALCOHOL,DISABILITY
3974,10/18/2008 8:00:00 AM,1500,Major Arterial,Scarborough,At Intersection,Traffic Signal,Clear,Daylight,Dry,Cyclist Collisions,...,No,No,No,No,No,No,Yes,No,No,No
11039,9/21/2014 8:00:00 AM,1542,Major Arterial,Scarborough,At/Near Private Drive,No Control,Clear,Daylight,Dry,Turning Movement,...,Yes,No,No,No,No,Yes,Yes,No,No,No
7841,1/17/2012 10:00:00 AM,1348,Major Arterial,Scarborough,Unknown,No Control,Rain,Daylight,Wet,Rear End,...,No,No,No,No,Yes,No,No,No,No,Yes
5638,3/1/2010 10:00:00 AM,1700,Major Arterial,Scarborough,At Intersection,Traffic Signal,Clear,Daylight,Dry,Turning Movement,...,No,No,No,No,Yes,No,Yes,No,No,No
5023,9/19/2009 8:00:00 AM,2315,Major Arterial,North York,At Intersection,Traffic Signal,Clear,Dark,Dry,Pedestrian Collisions,...,No,No,No,No,No,No,Yes,No,No,No


In [116]:
# change to pd date format
X_train['DATE'] = pd.to_datetime(X_train['DATE'])
X_test['DATE'] = pd.to_datetime(X_test['DATE'])

# 
X_train['HOUR'] = X_train['TIME'] // 100
X_test['HOUR'] = X_test['TIME'] // 100

In [117]:
# month=season mapping
season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
}
X_train['SEASON'] = X_train['DATE'].dt.month.map(season_map)
X_test['SEASON'] = X_test['DATE'].dt.month.map(season_map)

X_train['IS_WEEKEND'] = X_train['DATE'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
X_test['IS_WEEKEND'] = X_test['DATE'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# Time_Range
# -1 cause 00:00 will be included only in this way, pandas read (0, 5] NOT [0, 5]
bins = [-1, 5, 9, 14, 19, 24]
labels = ['Night', 'Morning_Rush', 'Day', 'Afternoon_Rush', 'Evening']

X_train['TIME_RANGE'] = pd.cut(X_train['HOUR'], bins=bins, labels=labels)
X_test['TIME_RANGE'] = pd.cut(X_test['HOUR'], bins=bins, labels=labels)

age_map = {
    '0 to 4': 'Child', '5 to 9': 'Child', '10 to 14': 'Child',
    '15 to 19': 'Young_Adult', '20 to 24': 'Young_Adult',
    '25 to 29': 'Adult', '30 to 34': 'Adult', '35 to 39': 'Adult',
    '40 to 44': 'Adult', '45 to 49': 'Adult', '50 to 54': 'Adult',
    '55 to 59': 'Adult', '60 to 64': 'Adult',
    '65 to 69': 'Senior', '70 to 74': 'Senior', '75 to 79': 'Senior',
    '80 to 84': 'Senior', '85 to 89': 'Senior', '90 to 94': 'Senior',
    'Over 95': 'Senior',
    'unknown': 'Unknown', 'Unknown': 'Unknown'
}
X_train['AGE_GROUP'] = X_train['INVAGE'].map(age_map)
X_test['AGE_GROUP'] = X_test['INVAGE'].map(age_map)

# if NaN after map() - change to Unknown
X_train['AGE_GROUP'] = X_train['AGE_GROUP'].fillna('Unknown')
X_test['AGE_GROUP'] = X_test['AGE_GROUP'].fillna('Unknown')

In [118]:
cols_to_drop = ['DATE', 'TIME', 'HOUR', 'INVAGE']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

display(X_train.head())

,ROAD_CLASS,DISTRICT,ACCLOC,TRAFFCTL,VISIBILITY,LIGHT,RDSFCOND,IMPACTYPE,INVTYPE,INITDIR,...,PASSENGER,SPEEDING,AG_DRIV,REDLIGHT,ALCOHOL,DISABILITY,SEASON,IS_WEEKEND,TIME_RANGE,AGE_GROUP
3974,Major Arterial,Scarborough,At Intersection,Traffic Signal,Clear,Daylight,Dry,Cyclist Collisions,Driver,South,...,No,No,Yes,No,No,No,Autumn,1,Afternoon_Rush,Adult
11039,Major Arterial,Scarborough,At/Near Private Drive,No Control,Clear,Daylight,Dry,Turning Movement,Motorcycle Driver,South,...,No,Yes,Yes,No,No,No,Autumn,1,Afternoon_Rush,Adult
7841,Major Arterial,Scarborough,Unknown,No Control,Rain,Daylight,Wet,Rear End,Driver,West,...,Yes,No,No,No,No,Yes,Winter,0,Day,Adult
5638,Major Arterial,Scarborough,At Intersection,Traffic Signal,Clear,Daylight,Dry,Turning Movement,Driver,East,...,Yes,No,Yes,No,No,No,Spring,0,Afternoon_Rush,Young_Adult
5023,Major Arterial,North York,At Intersection,Traffic Signal,Clear,Dark,Dry,Pedestrian Collisions,Pedestrian,West,...,No,No,Yes,No,No,No,Autumn,1,Evening,Young_Adult


In [119]:
# drop_first - remove duplicate column (IS_WEEKEND_1, IS_WEEKEND_0 = JUST IS_WEEKEND_1)
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# synch Train and Test
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

display(X_train_encoded.head())

,IS_WEEKEND,ROAD_CLASS_Expressway,ROAD_CLASS_Expressway Ramp,ROAD_CLASS_Laneway,ROAD_CLASS_Local,ROAD_CLASS_Major Arterial,ROAD_CLASS_Major Arterial,ROAD_CLASS_Major Shoreline,ROAD_CLASS_Minor Arterial,ROAD_CLASS_Other,...,SEASON_Summer,SEASON_Winter,TIME_RANGE_Morning_Rush,TIME_RANGE_Day,TIME_RANGE_Afternoon_Rush,TIME_RANGE_Evening,AGE_GROUP_Child,AGE_GROUP_Senior,AGE_GROUP_Unknown,AGE_GROUP_Young_Adult
3974,1,False,False,False,False,True,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
11039,1,False,False,False,False,True,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
7841,0,False,False,False,False,True,False,False,False,False,...,False,True,False,True,False,False,False,False,False,False
5638,0,False,False,False,False,True,False,False,False,False,...,False,False,False,False,True,False,False,False,False,True
5023,1,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True


In [124]:
print('Train:', X_train_encoded.shape)
print(y_train.shape)

print('Test:', X_test_encoded.shape)
print(y_test.shape)

Train: (15165, 234)
(15165,)
Test: (3792, 234)
(3792,)
